# AERO EYES — Pipeline Runner

**Hướng dẫn nhanh:**
1. `Runtime → Change runtime type → T4 GPU` (bắt buộc)
2. Chạy từng cell theo thứ tự từ trên xuống
3. Upload data của bạn vào Google Drive trước (xem hướng dẫn ở Section 2)

---
**Thông số nền tảng:**
| | Colab Free | Colab Pro | Kaggle Free |
|--|--|--|--|
| GPU | T4 (15 GB) | T4/A100 | T4 (16 GB) |
| RAM | 13 GB | 26 GB | 30 GB |
| Thời gian | ~12h/session | ~24h | 12h/week |
| Lưu file | Google Drive | Google Drive | Kaggle Dataset |

## Section 0 — Kiểm tra GPU

In [2]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('CẢNH BÁO: Không tìm thấy GPU!')
    print('Vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại.')

GPU: Tesla T4, 15360 MiB, 14913 MiB


## Section 1 — Clone Repo & Cài Dependencies

In [4]:
# ============================================================
# THAY BẰNG URL REPO CỦA BẠN
GITHUB_REPO = 'https://github.com/rhy221/aero_eyes_starter.git'
# ============================================================

import os
REPO_DIR = '/content/aero_eyes_starter'

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print('Repo đã tồn tại, pull latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

Cloning into '/content/aero_eyes_starter'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 48 (delta 0), reused 48 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 324.63 KiB | 10.14 MiB/s, done.
Working dir: /content/aero_eyes_starter


In [5]:
# Cài PyTorch (Colab thường đã có, cell này chỉ upgrade nếu cần)
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    print('Cài lại PyTorch với CUDA support...')
    !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

PyTorch 2.10.0+cu128, CUDA available: True


In [12]:
# %load /content/aero_eyes_starter/pyproject.toml
%%writefile /content/aero_eyes_starter/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "aero_eyes"
version = "0.1.0"
description = "Few-shot spatio-temporal drone object localization"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["."]
include = ["aero_eyes*"]

[tool.pytest.ini_options]
testpaths = ["tests"]



Overwriting /content/aero_eyes_starter/pyproject.toml


In [13]:
# Cài tất cả dependencies (~3-5 phút lần đầu)
!pip install -r requirements.txt -q
!pip install -e . -q
print('Done!')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for aero_eyes (pyproject.toml) ... done
Done!


In [37]:
import cv2
_has_tracker = hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerCSRT_create')
if not _has_tracker:
    print('Cài opencv-contrib để có tracker...')
    !pip install opencv-contrib-python-headless -q
    print('Tracker OK — khởi động lại kernel nếu import lỗi')
else:
    print(f'OpenCV {cv2.__version__} — tracker OK')

Cài opencv-contrib để có tracker...
Tracker OK — khởi động lại kernel nếu import lỗi


In [38]:
# Kiểm tra import
import numpy, cv2, yaml
from pydantic import BaseModel
from aero_eyes.config import load_config
from aero_eyes.types import Box
print('Tất cả imports OK!')
print(f'  numpy {numpy.__version__}, cv2 {cv2.__version__}')

Tất cả imports OK!
  numpy 2.0.2, cv2 4.11.0


## Section 2 — Mount Google Drive & Chuẩn Bị Data

**Cấu trúc thư mục cần có trong Google Drive của bạn:**
```
MyDrive/
└── aero_eyes_data/
    ├── annotations (1).json      ← file ground truth
    ├── Backpack_0/
    │   ├── refs/
    │   │   ├── ref_0.jpg
    │   │   ├── ref_1.jpg
    │   │   └── ref_2.jpg
    │   └── video.mp4
    └── Person_1/
        ├── refs/ ...
        └── video.mp4
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

Mounted at /content/drive
Google Drive mounted!


In [16]:
%%writefile /content/aero_eyes_starter/configs/config.yaml
# %load /content/aero_eyes_starter/configs/config.yaml
# ============================================================================
# AERO EYES — Few-Shot Spatio-Temporal Drone Object Localization
# Master configuration. Every switch here is a real, documented option.
# Override any field via CLI: --set proposal_model=fastsam_s
# ============================================================================

project:
  name: aero_eyes
  # Where intermediate per-stage artifacts + final outputs are written.
  work_dir: ./runs/exp001
  # Re-use cached artifacts (e.g. prototype) instead of recomputing.
  use_cache: true
  seed: 42

# ----------------------------------------------------------------------------
# DATA  —  CONFIRM THIS LAYOUT WITH THE USER BEFORE WRITING CODE.
# These are PLACEHOLDER assumptions; the real schema must be verified.
# ----------------------------------------------------------------------------
data:
  # One sample = 3 reference images + 1 drone video (+ optional GT).
  # Expected layout (TO BE CONFIRMED):
  #   data_root/
  #     <sample_id>/
  #       refs/   ref_0.jpg ref_1.jpg ref_2.jpg
  #       video.mp4
  #       gt.json            # optional, only for local eval
  data_root: ./data
  refs_subdir: object_images
  video_glob: "*.mp4"
  num_references: 3

  # Ground-truth schema — confirmed by user.
  # Single global annotations file at project root; filter by video_id.
  gt:
    global_file: "annotations (1).json"  # at project root, all videos
    box_format: xyxy          # confirmed: x1,y1,x2,y2 absolute pixels
    normalized: false
    frame_index_base: 0       # confirmed 0-based
    absent_encoding: omit     # confirmed: missing frames = absent
    one_object_per_video: true

  # Submission / output format — confirmed same schema as GT.
  submission:
    path_name: submission.json
    box_format: xyxy
    normalized: false
    frame_index_base: 0
    absent_encoding: omit

# ----------------------------------------------------------------------------
# RUNTIME
# ----------------------------------------------------------------------------
runtime:
  device: auto              # auto | cpu | cuda
  # Jetson/TensorRT explicitly OUT OF SCOPE. CPU/GPU portable only.
  num_workers: 4
  batch_size: 16            # candidate-feature extraction batch
  log_level: INFO
  save_visualizations: true # overlays/heatmaps/annotated video per stage

# ----------------------------------------------------------------------------
# STAGE 1 — Reference processing (offline, runs once per target)
# ----------------------------------------------------------------------------
stage1:
  segmentation:
    enabled: true
    model: mobilesam         # mobilesam  (background removal on refs)
    weights: null            # path to MobileSAM ckpt; null = auto-download
    fallback_if_missing: passthrough   # passthrough = skip masking, use full image
  feature_extractor:
    model: dinov2
    dinov2_variant: vitb14   # vits14 (~22M) | vitb14 (~86M, confirmed 8+GB VRAM)
    weights: null            # null = load from torch.hub / hf
    image_size: 224
  prototype:
    # How the 3 per-reference feature sets are fused.
    fusion: mean             # mean | max | concat_then_pca
    l2_normalize: true
    cache_name: prototype.npz

# ----------------------------------------------------------------------------
# STAGE 2 — Candidate generation (per keyframe)
# ----------------------------------------------------------------------------
stage2:
  keyframe_interval: 8       # sample 1 keyframe every N frames (5–10 typical)

  sahi:
    use_sahi: true           # <<< SAHI ON/OFF SWITCH
    tile: [640, 640]
    overlap: 0.25            # fraction
    # If false: run the proposal model on the full (resized) frame instead.

  proposal_model: yolov11n   # <<< "yolov11n"  |  "fastsam_s"  (pick exactly ONE)
                             #     YOLOv8 is explicitly NOT allowed.
  yolov11n:
    weights: yolo11n.pt      # ultralytics auto-downloads if absent
    conf: 0.05               # low: class-agnostic recall over precision
    iou: 0.5
    max_det: 300
    classes: null            # null = all classes (class-agnostic use)
  fastsam_s:
    weights: FastSAM-s.pt    # ultralytics auto-downloads if absent
    conf: 0.2
    iou: 0.7
    imgsz: 640

  candidate:
    min_box_area: 16         # discard sub-pixel noise (px^2)
    max_candidates_per_keyframe: 400
    feature_crop_pad: 0.10   # pad each candidate crop before DINOv2

# ----------------------------------------------------------------------------
# STAGE 3 — Cross-domain matching
# ----------------------------------------------------------------------------
stage3:
  similarity: cosine         # cosine
  match_threshold: 0.55      # calibratable; candidates below are dropped
  nms_iou: 0.5               # NMS across overlapping SAHI tiles
  topk_per_keyframe: 5       # keep at most K detections per keyframe
  # Optional auto-calibration of match_threshold on a labelled subset.
  calibrate:
    enabled: false
    target_metric: st_iou
    search_range: [0.40, 0.75]
    steps: 8

# ----------------------------------------------------------------------------
# STAGE 4 — Tracking between keyframes
# ----------------------------------------------------------------------------
stage4:
  tracker: builtin           # <<< "builtin" | "litetrack" | "none"
                             #   builtin   = OpenCV CSRT/KCF (no extra weights)
                             #   litetrack = LiteTrack-B4 ONNX (needs weights)
                             #   none      = run detection on EVERY frame
  builtin:
    algorithm: csrt          # csrt | kcf | mosse
  litetrack:
    onnx_path: null          # REQUIRED if tracker=litetrack; else fail clearly
    input_size: 256
  tracker_conf_threshold: 0.40   # τ: below this → re-detect immediately
  max_track_age: 30          # frames a track survives without re-detection

# ----------------------------------------------------------------------------
# STAGE 5 — Spatio-temporal output
# ----------------------------------------------------------------------------
stage5:
  temporal_smoothing:
    enabled: true
    method: ema              # ema | none
    ema_alpha: 0.6
  min_tube_length: 2         # discard tubes shorter than this (frames)
  fill_short_gaps: 3         # linearly interpolate gaps <= this many frames

# ----------------------------------------------------------------------------
# ACCURACY MODE — progressively enables domain-gap mitigations.
#   baseline       : single-scale, plain prototype
#   cheap_boosters : multi-scale scan + tuned NMS/thresh + multi-ref averaging
#   max_accuracy   : + synthetic viewpoint aug + CD-ViTO-style domain prompter
# Individual sub-toggles let each technique be ablated independently.
# ----------------------------------------------------------------------------
accuracy:
  mode: baseline             # baseline | cheap_boosters | max_accuracy

  cheap_boosters:
    multi_scale_scan: true
    scales: [0.75, 1.0, 1.5]
    tuned_nms: true
    multi_reference_embedding: true   # keep per-ref protos + ensemble match

  max_accuracy:
    synthetic_viewpoint_aug:
      enabled: true
      method: homography     # homography | perspective_warp
      num_synth_views: 6
      pitch_range_deg: [40, 85]   # simulate top-down
      fold_into_prototype: true
    domain_prompter:
      enabled: true
      # CD-ViTO-style: synthesize "imaginary domain" feature shifts and
      # align candidate features toward the reference style.
      num_prompts: 4
      strength: 0.3

# ----------------------------------------------------------------------------
# EVALUATION (ST-IoU)
# ----------------------------------------------------------------------------
eval:
  metric: st_iou
  # Per-frame IoU averaged over the TEMPORAL UNION of pred & GT tubes;
  # frames present in only one tube contribute IoU = 0.
  spatial_iou_type: standard
  report_per_video: true


Overwriting /content/aero_eyes_starter/configs/config.yaml


In [17]:
import os

# ============================================================
# CHỈNH ĐƯỜNG DẪN THEO CẤU TRÚC GOOGLE DRIVE CỦA BẠN
DRIVE_DATA_DIR = '/content/drive/MyDrive/AeroEyes/datasets/AeroEyes/observing/train/samples'
ANNOTATIONS_FILE = f'{DRIVE_DATA_DIR}/annotations (1).json'
# ============================================================

# Tạo symlink để pipeline tìm thấy data
DATA_LINK = f'{REPO_DIR}/data'
if not os.path.exists(DATA_LINK):
    os.symlink(DRIVE_DATA_DIR, DATA_LINK)
    print(f'Symlink tạo: {DATA_LINK} → {DRIVE_DATA_DIR}')
else:
    print(f'Symlink đã tồn tại: {DATA_LINK}')

# Symlink cho annotations file
ANN_LINK = f'{REPO_DIR}/annotations (1).json'
if not os.path.exists(ANN_LINK):
    os.symlink(ANNOTATIONS_FILE, ANN_LINK)

# Kiểm tra
samples = [d for d in os.listdir(DATA_LINK)
           if os.path.isdir(f'{DATA_LINK}/{d}')]
print(f'Tìm thấy {len(samples)} sample(s): {samples[:5]}')

Symlink tạo: /content/aero_eyes_starter/data → /content/drive/MyDrive/AeroEyes/datasets/AeroEyes/observing/train/samples
Tìm thấy 14 sample(s): ['Backpack_0', 'WaterBottle_0', 'Lifering_0', 'Lifering_1', 'Laptop_1']


## Section 3 — Cấu Hình Pipeline

In [27]:
# ============================================================
# CẤU HÌNH CHẠY — chỉnh theo nhu cầu
# ============================================================

# None = chạy TẤT CẢ sample; hoặc đặt tên cụ thể, ví dụ: 'Backpack_0'
SAMPLE_ID = None

# Accuracy mode: 'baseline' | 'cheap_boosters' | 'max_accuracy'
ACCURACY_MODE = 'cheap_boosters'

# Proposal model: 'yolov11n' | 'fastsam_s'
PROPOSAL_MODEL = 'fastsam_s'

# DINOv2 variant: 'vitb14' (tốt hơn, cần ~3GB VRAM) | 'vits14' (nhẹ hơn, cần ~1GB)
DINOV2_VARIANT = 'vitb14'   # T4 có 15GB nên dùng vitb14 được

# Batch size cho feature extraction
BATCH_SIZE = 16

# ── SAHI (Sliced Inference) ──────────────────────────────────
# Bật SAHI để detect vật thể nhỏ tốt hơn (slice frame thành nhiều tile nhỏ)
# True  = bật SAHI  → chậm hơn ~3-5x nhưng recall cao hơn với object nhỏ
# False = tắt SAHI  → nhanh hơn, dùng khi object vừa/lớn hoặc cần tốc độ
USE_SAHI = False

# Kích thước mỗi tile (pixels). Các giá trị phổ biến:
#   320  → nhiều tile hơn, detect object nhỏ hơn, chậm hơn
#   640  → cân bằng (default)
#   960  → ít tile hơn, nhanh hơn, bỏ sót object rất nhỏ
SAHI_TILE_SIZE = 640

# Overlap giữa các tile (0.0–0.5). 0.25 = overlap 25%
# Tăng lên 0.3-0.4 nếu bỏ sót object ở biên tile
SAHI_OVERLAP = 0.25
# ─────────────────────────────────────────────────────────────

# Thư mục lưu kết quả
WORK_DIR = '/content/drive/MyDrive/aero_eyes_runs/exp001'  # lưu thẳng vào Drive
# ============================================================

print('Cấu hình:')
print(f'  Sample:    {SAMPLE_ID or "ALL"}')
print(f'  Accuracy:  {ACCURACY_MODE}')
print(f'  Proposal:  {PROPOSAL_MODEL}')
print(f'  DINOv2:    {DINOV2_VARIANT}')
print(f'  SAHI:      {"ON" if USE_SAHI else "OFF"}  (tile={SAHI_TILE_SIZE}px, overlap={SAHI_OVERLAP})')
print(f'  Work dir:  {WORK_DIR}')

Cấu hình:
  Sample:    ALL
  Accuracy:  cheap_boosters
  Proposal:  fastsam_s
  DINOv2:    vitb14
  SAHI:      OFF  (tile=640px, overlap=0.25)
  Work dir:  /content/drive/MyDrive/aero_eyes_runs/exp001


In [28]:
# Tạo overrides list từ config ở trên
overrides = [
    f'project.work_dir={WORK_DIR}',
    f'data.data_root={DATA_LINK}',
    f'accuracy.mode={ACCURACY_MODE}',
    f'stage1.feature_extractor.dinov2_variant={DINOV2_VARIANT}',
    f'stage2.proposal_model={PROPOSAL_MODEL}',
    f'runtime.batch_size={BATCH_SIZE}',
    f'stage2.sahi.use_sahi={str(USE_SAHI).lower()}',
    f'stage2.sahi.tile=[{SAHI_TILE_SIZE},{SAHI_TILE_SIZE}]',
    f'stage2.sahi.overlap={SAHI_OVERLAP}',
]
overrides_str = ' '.join(f'--set {o}' for o in overrides)
SAMPLE_STR = f'--sample {SAMPLE_ID}' if SAMPLE_ID else ''

print('CLI overrides:', overrides_str)

CLI overrides: --set project.work_dir=/content/drive/MyDrive/aero_eyes_runs/exp001 --set data.data_root=/content/aero_eyes_starter/data --set accuracy.mode=cheap_boosters --set stage1.feature_extractor.dinov2_variant=vitb14 --set stage2.proposal_model=fastsam_s --set runtime.batch_size=16 --set stage2.sahi.use_sahi=false --set stage2.sahi.tile=[640,640] --set stage2.sahi.overlap=0.25


## Section 4 — Chạy Pipeline

In [31]:
%%writefile /content/aero_eyes_starter/aero_eyes/config.py
# %load /content/aero_eyes_starter/aero_eyes/config.py
"""Typed configuration schema + loader.

Loads configs/config.yaml into validated Pydantic models.
Supports CLI overrides:  --set stage2.proposal_model=fastsam_s
"""
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Literal, Optional

import yaml
from pydantic import BaseModel, field_validator, model_validator


# ---------------------------------------------------------------------------
# Sub-models
# ---------------------------------------------------------------------------

class ProjectConfig(BaseModel):
    name: str = "aero_eyes"
    work_dir: str = "./runs/exp001"
    use_cache: bool = True
    seed: int = 42


class GTConfig(BaseModel):
    global_file: str = "annotations (1).json"
    box_format: Literal["xyxy", "xywh", "cxcywh"] = "xyxy"
    normalized: bool = False
    frame_index_base: int = 0
    absent_encoding: Literal["omit", "null_box", "empty_list"] = "omit"
    one_object_per_video: bool = True


class SubmissionConfig(BaseModel):
    path_name: str = "submission.json"
    box_format: Literal["xyxy", "xywh", "cxcywh"] = "xyxy"
    normalized: bool = False
    frame_index_base: int = 0
    absent_encoding: Literal["omit", "null_box", "empty_list"] = "omit"


class DataConfig(BaseModel):
    data_root: str = "./data"
    refs_subdir: str = "refs"
    video_glob: str = "*.mp4"
    num_references: int = 3
    gt: GTConfig = GTConfig()
    submission: SubmissionConfig = SubmissionConfig()


class RuntimeConfig(BaseModel):
    device: str = "auto"
    num_workers: int = 4
    batch_size: int = 16
    log_level: str = "INFO"
    save_visualizations: bool = True


class SegmentationConfig(BaseModel):
    enabled: bool = True
    model: str = "mobilesam"
    weights: Optional[str] = None
    fallback_if_missing: str = "passthrough"


class FeatureExtractorConfig(BaseModel):
    model: str = "dinov2"
    dinov2_variant: Literal["vits14", "vitb14"] = "vitb14"
    weights: Optional[str] = None
    image_size: int = 224


class PrototypeConfig(BaseModel):
    fusion: Literal["mean", "max", "concat_then_pca"] = "mean"
    l2_normalize: bool = True
    cache_name: str = "prototype.npz"


class Stage1Config(BaseModel):
    segmentation: SegmentationConfig = SegmentationConfig()
    feature_extractor: FeatureExtractorConfig = FeatureExtractorConfig()
    prototype: PrototypeConfig = PrototypeConfig()


class SAHIConfig(BaseModel):
    use_sahi: bool = True
    tile: list[int] = [640, 640]
    overlap: float = 0.25


class Yolov11nConfig(BaseModel):
    weights: str = "yolo11n.pt"
    conf: float = 0.05
    iou: float = 0.5
    max_det: int = 300
    classes: Optional[Any] = None


class FastSamSConfig(BaseModel):
    weights: str = "FastSAM-s.pt"
    conf: float = 0.2
    iou: float = 0.7
    imgsz: int = 640


class CandidateConfig(BaseModel):
    min_box_area: float = 16.0
    max_candidates_per_keyframe: int = 400
    feature_crop_pad: float = 0.10


class Stage2Config(BaseModel):
    keyframe_interval: int = 8
    sahi: SAHIConfig = SAHIConfig()
    proposal_model: str = "yolov11n"
    yolov11n: Yolov11nConfig = Yolov11nConfig()
    fastsam_s: FastSamSConfig = FastSamSConfig()
    candidate: CandidateConfig = CandidateConfig()

    @field_validator("proposal_model")
    @classmethod
    def check_proposal_model(cls, v: str) -> str:
        allowed = {"yolov11n", "fastsam_s"}
        if v not in allowed:
            raise ValueError(
                f"stage2.proposal_model must be one of {allowed}; got '{v}'. "
                "YOLOv8 is explicitly NOT allowed."
            )
        return v


class CalibrateConfig(BaseModel):
    enabled: bool = False
    target_metric: str = "st_iou"
    search_range: list[float] = [0.40, 0.75]
    steps: int = 8


class Stage3Config(BaseModel):
    similarity: str = "cosine"
    match_threshold: float = 0.55
    nms_iou: float = 0.5
    topk_per_keyframe: int = 5
    calibrate: CalibrateConfig = CalibrateConfig()


class BuiltinTrackerConfig(BaseModel):
    algorithm: Literal["csrt", "kcf", "mosse"] = "csrt"


class LiteTrackConfig(BaseModel):
    onnx_path: Optional[str] = None
    input_size: int = 256


class Stage4Config(BaseModel):
    tracker: str = "builtin"
    builtin: BuiltinTrackerConfig = BuiltinTrackerConfig()
    litetrack: LiteTrackConfig = LiteTrackConfig()
    tracker_conf_threshold: float = 0.40
    max_track_age: int = 30

    @field_validator("tracker")
    @classmethod
    def check_tracker(cls, v: str) -> str:
        allowed = {"builtin", "litetrack", "none"}
        if v not in allowed:
            raise ValueError(f"stage4.tracker must be one of {allowed}; got '{v}'.")
        return v


class TemporalSmoothingConfig(BaseModel):
    enabled: bool = True
    method: Literal["ema", "none"] = "ema"
    ema_alpha: float = 0.6


class Stage5Config(BaseModel):
    temporal_smoothing: TemporalSmoothingConfig = TemporalSmoothingConfig()
    min_tube_length: int = 2
    fill_short_gaps: int = 3


class SyntheticViewpointAugConfig(BaseModel):
    enabled: bool = True
    method: Literal["homography", "perspective_warp"] = "homography"
    num_synth_views: int = 6
    pitch_range_deg: list[float] = [40.0, 85.0]
    fold_into_prototype: bool = True


class DomainPrompterConfig(BaseModel):
    enabled: bool = True
    num_prompts: int = 4
    strength: float = 0.3


class CheapBoostersConfig(BaseModel):
    multi_scale_scan: bool = True
    scales: list[float] = [0.75, 1.0, 1.5]
    tuned_nms: bool = True
    multi_reference_embedding: bool = True


class MaxAccuracyConfig(BaseModel):
    synthetic_viewpoint_aug: SyntheticViewpointAugConfig = SyntheticViewpointAugConfig()
    domain_prompter: DomainPrompterConfig = DomainPrompterConfig()


class AccuracyConfig(BaseModel):
    mode: Literal["baseline", "cheap_boosters", "max_accuracy"] = "baseline"
    cheap_boosters: CheapBoostersConfig = CheapBoostersConfig()
    max_accuracy: MaxAccuracyConfig = MaxAccuracyConfig()


class EvalConfig(BaseModel):
    metric: str = "st_iou"
    spatial_iou_type: str = "standard"
    report_per_video: bool = True


# ---------------------------------------------------------------------------
# Top-level config
# ---------------------------------------------------------------------------

class AeroEyesConfig(BaseModel):
    project: ProjectConfig = ProjectConfig()
    data: DataConfig = DataConfig()
    runtime: RuntimeConfig = RuntimeConfig()
    stage1: Stage1Config = Stage1Config()
    stage2: Stage2Config = Stage2Config()
    stage3: Stage3Config = Stage3Config()
    stage4: Stage4Config = Stage4Config()
    stage5: Stage5Config = Stage5Config()
    accuracy: AccuracyConfig = AccuracyConfig()
    eval: EvalConfig = EvalConfig()

    @model_validator(mode="after")
    def check_litetrack_path(self) -> "AeroEyesConfig":
        if self.stage4.tracker == "litetrack":
            if not self.stage4.litetrack.onnx_path:
                raise ValueError(
                    "stage4.tracker is 'litetrack' but stage4.litetrack.onnx_path is not set. "
                    "Download the LiteTrack-B4 ONNX weights and set "
                    "stage4.litetrack.onnx_path=/path/to/litetrack.onnx in your config."
                )
        return self

    def sample_work_dir(self, sample_id: str) -> Path:
        return Path(self.project.work_dir) / sample_id

    def device(self) -> str:
        if self.runtime.device != "auto":
            return self.runtime.device
        try:
            import torch
            return "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            return "cpu"


# ---------------------------------------------------------------------------
# Loader
# ---------------------------------------------------------------------------

def _deep_merge(base: dict, override: dict) -> dict:
    """Recursively merge override into base."""
    result = dict(base)
    for k, v in override.items():
        if k in result and isinstance(result[k], dict) and isinstance(v, dict):
            result[k] = _deep_merge(result[k], v)
        else:
            result[k] = v
    return result


def _parse_override(s: str) -> tuple[list[str], str]:
    """Parse 'a.b.c=value' into (['a','b','c'], 'value')."""
    m = re.match(r"^([\w.]+)=(.*)$", s, re.DOTALL)
    if not m:
        raise ValueError(f"Invalid override '{s}'; expected dotted.key=value")
    keys = m.group(1).split(".")
    raw = m.group(2)
    # Try to coerce to Python primitive types
    if raw.lower() == "true":
        value: Any = True
    elif raw.lower() == "false":
        value = False
    elif raw.lower() in ("null", "none", "~"):
        value = None
    else:
        try:
            value = int(raw)
        except ValueError:
            try:
                value = float(raw)
            except ValueError:
                # Try JSON (handles lists like [640,640] and dicts)
                if raw.startswith(("[", "{")):
                    try:
                        import json as _json
                        value = _json.loads(raw)
                    except Exception:
                        value = raw
                else:
                    value = raw
    return keys, value


def _set_nested(d: dict, keys: list[str], value: Any) -> None:
    for k in keys[:-1]:
        d = d.setdefault(k, {})
    d[keys[-1]] = value


def load_config(path: str | Path, overrides: list[str] | None = None) -> AeroEyesConfig:
    """Load config.yaml, apply CLI overrides, validate and return typed config."""
    with open(path) as f:
        raw: dict = yaml.safe_load(f) or {}

    if overrides:
        for ov in overrides:
            keys, value = _parse_override(ov)
            _set_nested(raw, keys, value)

    return AeroEyesConfig.model_validate(raw)


Overwriting /content/aero_eyes_starter/aero_eyes/config.py


In [35]:
%%writefile /content/aero_eyes_starter/aero_eyes/models/features.py
# %load /content/aero_eyes_starter/aero_eyes/models/features.py
"""DINOv2 feature extractor — ViT-S/14 or ViT-B/14.

Loads via torch.hub (facebookresearch/dinov2) with fallback to HuggingFace.
Returns L2-normalized CLS token features. CPU/GPU auto-detect; batched.
"""
from __future__ import annotations

import logging
from typing import Any

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from aero_eyes.types import Box
from aero_eyes.utils.geometry import crop_with_pad

log = logging.getLogger(__name__)

# ImageNet normalization (what DINOv2 expects)
_MEAN = (0.485, 0.456, 0.406)
_STD = (0.229, 0.224, 0.225)


def _preprocess(img_bgr: np.ndarray, image_size: int = 224) -> torch.Tensor:
    """BGR uint8 ndarray -> normalized float32 tensor [3, H, W]."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb).resize((image_size, image_size), Image.BICUBIC)
    mean = np.array(_MEAN, dtype=np.float32)
    std = np.array(_STD, dtype=np.float32)
    arr = np.array(img_pil, dtype=np.float32) / 255.0
    arr = (arr - mean) / std
    return torch.from_numpy(arr.transpose(2, 0, 1).copy())  # [3, H, W] float32


class DINOv2FeatureExtractor:
    """Batched DINOv2 feature extractor returning L2-normalized CLS tokens."""

    def __init__(self, variant: str = "vitb14", device: str = "auto", image_size: int = 224):
        self.variant = variant
        self.image_size = image_size
        if device == "auto":
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device

        self.model = self._load_model(variant)
        self.model.eval()
        self.model.to(self.device)
        log.info("DINOv2 %s loaded on %s", variant, self.device)

    def _load_model(self, variant: str) -> Any:
        model_name = f"dinov2_{variant}"
        try:
            model = torch.hub.load(
                "facebookresearch/dinov2", model_name, pretrained=True
            )
            log.debug("DINOv2 loaded via torch.hub")
            return model
        except Exception as e:
            log.warning("torch.hub load failed (%s), trying HuggingFace", e)
        try:
            from transformers import AutoModel
            hf_name = f"facebook/dinov2-{variant.replace('vit', 'vit-').replace('14', '-patch14')}"
            # Map variant names: vits14 -> vit-small-patch14, vitb14 -> vit-base-patch14
            hf_map = {
                "vits14": "facebook/dinov2-small",
                "vitb14": "facebook/dinov2-base",
            }
            hf_name = hf_map.get(variant, f"facebook/dinov2-{variant}")
            model = AutoModel.from_pretrained(hf_name)
            model._hf_mode = True
            return model
        except Exception as e2:
            raise RuntimeError(
                f"Could not load DINOv2 '{variant}' via torch.hub or HuggingFace. "
                f"torch.hub error: {e}. HuggingFace error: {e2}. "
                "Check your internet connection or manually download the weights."
            ) from e2

    @torch.no_grad()
    def extract(self, images: list[np.ndarray], batch_size: int = 16) -> np.ndarray:
        """Extract L2-normalized CLS features from a list of BGR uint8 images.

        Returns np.ndarray [N, D].
        """
        if not images:
            return np.zeros((0, self._feature_dim()), dtype=np.float32)

        tensors = [_preprocess(img, self.image_size) for img in images]
        all_feats: list[np.ndarray] = []

        for i in range(0, len(tensors), batch_size):
            batch = torch.stack(tensors[i:i + batch_size]).to(self.device).float()
            if hasattr(self.model, "_hf_mode") and self.model._hf_mode:
                from transformers import AutoFeatureExtractor
                out = self.model(pixel_values=batch)
                feats = out.last_hidden_state[:, 0]  # CLS token
            else:
                feats = self.model(batch)  # DINOv2 returns CLS directly
            feats = F.normalize(feats, dim=-1)
            all_feats.append(feats.cpu().numpy())

        return np.concatenate(all_feats, axis=0).astype(np.float32)

    def extract_crops(
        self,
        frame_bgr: np.ndarray,
        boxes: list[Box],
        pad_ratio: float = 0.10,
        batch_size: int = 16,
    ) -> np.ndarray:
        """Crop each box from frame_bgr and extract features. Returns [N, D]."""
        if not boxes:
            return np.zeros((0, self._feature_dim()), dtype=np.float32)
        crops = [crop_with_pad(frame_bgr, b, pad_ratio) for b in boxes]
        return self.extract(crops, batch_size=batch_size)

    def _feature_dim(self) -> int:
        _dims = {"vits14": 384, "vitb14": 768}
        return _dims.get(self.variant, 768)



Overwriting /content/aero_eyes_starter/aero_eyes/models/features.py


In [39]:
# Chạy TOÀN BỘ pipeline (Stage 1 → 5)
!python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    {SAMPLE_STR} \
    {overrides_str}

INFO:__main__:=== Running pipeline for sample: Backpack_0 (from stage 1) ===
INFO:__main__:--- Stage 1 ---
INFO:aero_eyes.stages.stage1:[Stage1] Backpack_0: using cached prototype at /content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_0/prototype.npz
INFO:__main__:--- Stage 2 ---
INFO:aero_eyes.stages.stage2:[Stage2] Backpack_0: using cached candidates at /content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_0/candidates.json
INFO:__main__:--- Stage 3 ---
INFO:aero_eyes.stages.stage3:[Stage3] Backpack_0: using cached detections at /content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_0/detections.json
INFO:__main__:--- Stage 4 ---
INFO:aero_eyes.stages.stage4:[Stage4] Backpack_0: using cached tracks at /content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_0/tracks.json
INFO:__main__:--- Stage 5 ---
INFO:aero_eyes.stages.stage5:[Stage5] Backpack_0: 0 frames with boxes before smoothing
INFO:aero_eyes.stages.stage5:[Stage5] Backpack_0: 0 frames in final tube
INFO:aero_eyes.utils.io:Wrot

In [ ]:
# --- HOẶC chạy từng stage riêng để dễ debug ---

# Thay SAMPLE_ID bằng tên sample muốn chạy
S = SAMPLE_ID or samples[0]  # dùng sample đầu tiên nếu không chỉ định
print(f'Chạy từng stage cho sample: {S}')

In [ ]:
# Stage 1 — Reference processing → prototype.npz
!python -m aero_eyes.stages.stage1 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 2 — Candidate generation → candidates.json  (lâu nhất)
!python -m aero_eyes.stages.stage2 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 3 — Cross-domain matching → detections.json
!python -m aero_eyes.stages.stage3 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 4 — Tracking → tracks.json
!python -m aero_eyes.stages.stage4 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 5 — Spatio-temporal output → submission.json
!python -m aero_eyes.stages.stage5 --config configs/config.yaml --sample {S} {overrides_str}

## Section 5 — Đánh Giá & Xem Kết Quả

In [42]:
import json, os
import numpy as np

sample = 'Backpack_0'  # ← đổi thành sample bị fail
work = f'/content/drive/MyDrive/aero_eyes_runs/exp001/{sample}'

det = json.load(open(f'{work}/detections.json'))
cand = json.load(open(f'{work}/candidates.json'))

# Số keyframe có detection vs tổng
print(f'Keyframes có detection: {len(det)} / {len(cand)}')

# Xem similarity của candidates (từ stage 2 feature file)
import numpy as np
feats = np.load(f'{work}/candidates.feats.npz', allow_pickle=True)
proto = np.load(f'{work}/prototype.npz')['prototype']

sims = feats['features'] @ proto  # cosine (đã L2-norm)
print(f'Similarity  max={sims.max():.3f}  mean={sims.mean():.3f}  p95={np.percentile(sims,95):.3f}')
print(f'Candidates vượt threshold 0.55: {(sims > 0.55).sum()} / {len(sims)}')
print(f'Candidates vượt threshold 0.35: {(sims > 0.35).sum()} / {len(sims)}')
print(f'Candidates vượt threshold 0.25: {(sims > 0.25).sum()} / {len(sims)}')


Keyframes có detection: 2 / 2
Similarity  max=0.312  mean=0.010  p95=0.063
Candidates vượt threshold 0.55: 0 / 9325
Candidates vượt threshold 0.35: 0 / 9325
Candidates vượt threshold 0.25: 9 / 9325


In [40]:
import os, json

# Liệt kê tất cả submission.json đã tạo
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == 'submission.json':
            path = os.path.join(root, f)
            with open(path) as fp:
                data = json.load(fp)
            n_frames = len(data[0]['annotations'][0]['bboxes']) if data else 0
            print(f'{path}  ({n_frames} frames with detection)')

/content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_0/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Backpack_1/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Jacket_0/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Jacket_1/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Laptop_0/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Laptop_1/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Lifering_0/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/Lifering_1/submission.json  (1371 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/MobilePhone_0/submission.json  (0 frames with detection)
/content/drive/MyDrive/aero_eyes_runs/exp001/MobilePhone_1/submission.json  (0 frames with detection)
/content/

In [41]:
# Tính ST-IoU (cần có ground truth)
GT_FILE = ANN_LINK  # annotations (1).json

# Gom tất cả submission thành 1 file để eval toàn bộ
all_preds = []
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == 'submission.json':
            with open(os.path.join(root, f)) as fp:
                all_preds.extend(json.load(fp))

COMBINED_PRED = '/content/all_submissions.json'
with open(COMBINED_PRED, 'w') as fp:
    json.dump(all_preds, fp)
print(f'Gom {len(all_preds)} video vào {COMBINED_PRED}')

!python -m aero_eyes.evaluate \
    --pred {COMBINED_PRED} \
    --gt "{GT_FILE}" \
    --config configs/config.yaml

Gom 14 video vào /content/all_submissions.json
INFO:__main__:  Backpack_0                     ST-IoU = 0.0000
INFO:__main__:  Backpack_1                     ST-IoU = 0.0000
INFO:__main__:  Jacket_0                       ST-IoU = 0.0000
INFO:__main__:  Jacket_1                       ST-IoU = 0.0000
INFO:__main__:  Laptop_0                       ST-IoU = 0.0000
INFO:__main__:  Laptop_1                       ST-IoU = 0.0000
INFO:__main__:  Lifering_0                     ST-IoU = 0.0000
INFO:__main__:  Lifering_1                     ST-IoU = 0.5805
INFO:__main__:  MobilePhone_0                  ST-IoU = 0.0000
INFO:__main__:  MobilePhone_1                  ST-IoU = 0.0000
INFO:__main__:  Person1_0                      ST-IoU = 0.0000
INFO:__main__:  Person1_1                      ST-IoU = 0.0000
INFO:__main__:  WaterBottle_0                  ST-IoU = 0.0000
INFO:__main__:  WaterBottle_1                  ST-IoU = 0.0000

Mean ST-IoU: 0.0415

Per-video scores:
  Backpack_0: 0.0000
  Backpack

In [ ]:
# Tải file submission về máy local (nếu cần)
from google.colab import files
files.download(COMBINED_PRED)

## Section 6 — Chạy trên Dữ Liệu Giả (không cần dataset thật)

Dùng để test nhanh pipeline khi chưa có data thật.

In [18]:
# Tạo synthetic fixture
!python -m scripts.make_synthetic_fixture --out tests/fixtures

# Chạy pipeline trên fixture
!python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --sample synth001 \
    --set data.data_root=tests/fixtures \
    --set project.work_dir=runs/synth_test

# Đánh giá
!python -m aero_eyes.evaluate \
    --pred runs/synth_test/synth001/submission.json \
    --gt tests/fixtures/synth001/gt.json \
    --config configs/config.yaml

# Chạy unit tests
!python -m pytest tests/test_st_iou.py -v

Synthetic fixture created at tests/fixtures/synth001
  3 reference images in tests/fixtures/synth001/refs
  video: tests/fixtures/synth001/video.mp4 (30 frames, object in frames 5-25)
  GT: tests/fixtures/synth001/gt.json (21 annotated frames)
INFO:__main__:=== Running pipeline for sample: synth001 (from stage 1) ===
INFO:__main__:--- Stage 1 ---
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/aero_eyes_starter/aero_eyes/stages/run_all.py", line 66, in <module>
    main()
  File "/content/aero_eyes_starter/aero_eyes/stages/run_all.py", line 62, in main
    run_all(cfg, sample_id=args.sample, from_stage=args.from_stage)
  File "/content/aero_eyes_starter/aero_eyes/stages/run_all.py", line 47, in run_all
    fn(cfg, sid)
  File "/content/aero_eyes_starter/aero_eyes/stages/stage1.py", line 48, in run_stage1
    raise FileNotFoundError(
FileNotFoundError: Expected 3 reference images